# Урок 12 - Намаляване на историята на чата с помощта на Agent Scratchpad

Този бележник демонстрира как да управлявате контекста при дълги разговори с използване на Microsoft Agent Framework. С увеличаването на разговорите броят токени се увеличава — в крайна сметка превишава контекстното поле на модела. Ние решаваме това с помощта на **шаблон за обобщаване на контекста** и **agent scratchpad** за постоянна памет.

## Какво ще научите:
1. **Защо е важно управлението на контекста**: Разбиране на ограниченията на токените и контекстните полета
2. **Агенти, осведомени за контекста**: Създаване на агенти, които управляват собствения си контекст на разговора
3. **Шаблон за обобщаване на контекста**: Използване на инструменти за кондензиране на историята на разговора
4. **Agent Scratchpad**: Постоянна памет, която оцелява през намаляването на контекста

## Предварителни изисквания:
- Настройка на Azure OpenAI с конфигурирани променливи на средата
- Разбиране на основните концепции за агенти от предишните уроци


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Защо управлението на контекста е важно

Всеки LLM има ограничено **контекстно прозорче** — максималният брой токени, които може да обработи в една заявка. С напредването на многотуровия разговор:

- **Броят на токените расте линейно** с всяко потребителско съобщение и отговор на асистента.
- **Токените на промпта доминират в разходите**, защото цялата история се изпраща отново на всяка стъпка.
- Накрая разговорът **превишава контекстното прозорче** и моделът или отрязва, или връща грешка.

### Стратегии за управление на контекста

| Стратегия | Как работи | Компромис |
|---|---|---|
| **Отрязване** | Пропуска най-старите съобщения | Загуба на ранния контекст |
| **Обобщение** | Съкращава по-старите съобщения в обобщение | Някои подробности се губят, но ключовите точки се запазват |
| **Бележник / Външна памет** | Съхранява ключови факти извън разговора | Изисква извикване на инструменти, но оцелява при всяко съкращение |

В този тетрадка комбинираме **обобщение** с **инструмент за бележник**, така че агентът да може да поддържа непрекъснатост дори когато историята на разговора е свита.


## Създаване на агент, осъзнаващ контекста


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Симулиране на дълъг разговор

Нека преминем през многообменен разговор, за да видим как контекстът се натрупва. Агентът трябва да запази ключови детайли (предпочитания, бюджет, дати на пътуване) през ходовете и да демонстрира непрекъснатост.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Обърнете внимание как агентът запазва контекста от по-ранните реплики — той знае за Япония, суши, храмове, фотография, бюджета от 3000 долара, самотното пътуване и времевия период през април. В кратък разговор това работи добре, но с разрастването на разговора пълната история става скъпа за повторно изпращане.

Нека продължим разговора с още реплики, за да видим натрупването на контекст:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Контекстен модел за обобщение

С нарастването на разговора можем да използваме **инструмент за обобщение**, за да кондензираме натрупания контекст в компактен формат. Агентът извиква този инструмент, за да запише ключови предпочитания, така че дори по-старите съобщения да бъдат премахнати, съществената информация да се запази.

Този модел е градивен елемент за по-сложни редукции на историята:
1. Агентът идентифицира ключови факти от разговора
2. Извиква инструмента за обобщение, за да ги съхрани
3. По-старите съобщения могат да бъдат безопасно премахнати, защото обобщението улавя важната информация

По-долу дефинираме инструмент `summarize_preferences`, който агентът може да извика, за да запише компактен обобщен преглед на това, което е научил.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Обобщение

В този урок научихте как да управлявате контекста в дългосрочни разговори с агент, използвайки Microsoft Agent Framework:

### Ключови концепции
- **Контекстните прозорци са ограничени** — всеки токен в историята на разговора струва пари и се брои към лимита.
- **Инструментите за обобщаване** позволяват на агента да кондензира натрупания контекст в компактни обобщения, намалявайки използването на токени, при запазване на съществената информация.
- **Бележниците на агента** предоставят постоянна външна памет, която оцелява при всяко намаляване на разговора.

### Какво изградихте
- **Агент, осъзнаващ контекста**, който поддържа непрекъснатост в многоходови разговори
- **Инструмент за обобщаване** (`summarize_preferences`), който записва ключови детайли за потребителя в компактен формат
- **Многоходов разговор**, демонстриращ задържане на контекста и обработка на промени

### Приложения в реалния свят
- **Ботове за обслужване на клиенти**: Запомняне на предпочитания през дълги сесии за поддръжка
- **Лични асистенти**: Проследяване на текущи проекти без необходимост от обясняване на контекста отново
- **Образователни учители**: Поддържане на напредъка на ученика през множество взаимодействия

### Следващи стъпки
- Имплементиране на пълен бележник с файлово запазване
- Добавяне на автоматично съкращаване на историята след обобщаване
- Комбиниране с векторни бази данни за семантично търсене в паметта
- Създаване на агенти, които могат да възобновят разговори дни по-късно с пълен контекст


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля, имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да било недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
